# 09. Advanced attention — Kimi K3 attention path at reduced tensor scale

This notebook keeps the **K3 attention topology and depth** while reducing only tensor dimensions for a CPU sanity check.

Kept from K3:

- 93 decoder layers
- 69 KDA + 24 Gated MLA schedule
- KDA short convolution, low-rank decay parameterization, lower-bounded decay, delta-rule state update, head-wise RMSNorm, and full-rank output gate
- Gated MLA low-rank Q/KV paths, NoPE attention, and output gate
- Block Attention Residuals with block size 12, separate attention/FFN pseudo-queries, and the final output residual mix

Reduced only for execution: hidden width, head count/head width, low-rank widths, sequence length, and batch size. The MoE subsystem is studied separately in notebook 10, so the FFN here uses the K3 SiTU-GLU dense computation to keep the attention/AttnRes path readable.


In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(11)
torch.set_num_threads(min(2, torch.get_num_threads()))
device = torch.device("cpu")
print("device:", device)


## 1. Basic normalization and KDA recurrence

KDA first decays the matrix state channel-wise, predicts the value stored under the current key, writes the delta error, and then reads with the query. The recurrence order is kept explicit rather than replaced by generic attention.


In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        scale = torch.rsqrt(x.square().mean(dim=-1, keepdim=True) + self.eps)
        return x * scale * self.weight


def kda_scan(q, k, v, alpha, beta):
    batch, heads, length, key_dim = q.shape
    value_dim = v.size(-1)

    state = torch.zeros(
        batch,
        heads,
        key_dim,
        value_dim,
        device=q.device,
        dtype=q.dtype,
    )
    outputs = []

    for time_index in range(length):
        q_t = F.normalize(q[:, :, time_index], dim=-1)
        k_t = F.normalize(k[:, :, time_index], dim=-1)
        v_t = v[:, :, time_index]

        decayed_state = alpha[:, :, time_index, :, None] * state
        predicted_value = torch.einsum(
            "bhkv,bhk->bhv",
            decayed_state,
            k_t,
        )
        error = v_t - predicted_value

        state = (
            decayed_state
            + beta[:, :, time_index, None, None]
            * torch.einsum("bhk,bhv->bhkv", k_t, error)
        )
        output_t = torch.einsum(
            "bhkv,bhk->bhv",
            state,
            q_t,
        )
        outputs.append(output_t)

    return torch.stack(outputs, dim=2)


## 2. KDA with the K3 decay and output-gate parameterization

The decay pre-activation keeps the low-rank `f_a -> f_b` path. For each head, a learned positive scale `exp(A_h)` enters the lower-bounded gate

`g_t = g_min * sigmoid(exp(A_h) * z_t)`, `alpha_t = exp(g_t)`.

The output gate is full rank, as in K3.


In [ ]:
class TinyKDA(nn.Module):
    def __init__(
        self,
        model_dim=8,
        heads=2,
        decay_rank=2,
        short_kernel=4,
    ):
        super().__init__()
        assert model_dim % heads == 0

        self.model_dim = model_dim
        self.heads = heads
        self.head_dim = model_dim // heads
        self.short_kernel = short_kernel
        self.gate_lower_bound = -5.0

        self.q_proj = nn.Linear(model_dim, model_dim, bias=False)
        self.k_proj = nn.Linear(model_dim, model_dim, bias=False)
        self.v_proj = nn.Linear(model_dim, model_dim, bias=False)

        self.q_conv = nn.Conv1d(
            model_dim, model_dim, short_kernel, groups=model_dim
        )
        self.k_conv = nn.Conv1d(
            model_dim, model_dim, short_kernel, groups=model_dim
        )
        self.v_conv = nn.Conv1d(
            model_dim, model_dim, short_kernel, groups=model_dim
        )

        self.f_a_proj = nn.Linear(model_dim, decay_rank, bias=False)
        self.f_b_proj = nn.Linear(decay_rank, model_dim, bias=True)
        self.beta_proj = nn.Linear(model_dim, heads, bias=True)
        self.A_log = nn.Parameter(torch.zeros(heads))

        self.output_norm = RMSNorm(self.head_dim)
        self.output_gate = nn.Linear(model_dim, model_dim, bias=True)
        self.out_proj = nn.Linear(model_dim, model_dim, bias=False)

    def _causal_short_conv(self, x, convolution):
        x = x.transpose(1, 2)
        x = F.pad(x, (self.short_kernel - 1, 0))
        x = convolution(x).transpose(1, 2)
        return F.silu(x)

    def _split_heads(self, x):
        batch, length, _ = x.shape
        return x.view(
            batch, length, self.heads, self.head_dim
        ).transpose(1, 2)

    def forward(self, hidden):
        q = self._split_heads(
            self._causal_short_conv(self.q_proj(hidden), self.q_conv)
        )
        k = self._split_heads(
            self._causal_short_conv(self.k_proj(hidden), self.k_conv)
        )
        v = self._split_heads(
            self._causal_short_conv(self.v_proj(hidden), self.v_conv)
        )

        decay_pre = self.f_b_proj(
            F.silu(self.f_a_proj(hidden))
        )
        decay_pre = self._split_heads(decay_pre)

        head_scale = torch.exp(self.A_log)[None, :, None, None]
        log_decay = self.gate_lower_bound * torch.sigmoid(
            head_scale * decay_pre
        )
        alpha = torch.exp(log_decay)
        beta = torch.sigmoid(self.beta_proj(hidden)).transpose(1, 2)

        scanned = kda_scan(q, k, v, alpha, beta)
        scanned = self.output_norm(scanned)
        scanned = scanned.transpose(1, 2).contiguous().reshape_as(hidden)

        gate = torch.sigmoid(self.output_gate(hidden))
        return self.out_proj(gate * scanned)


## 3. Gated MLA, NoPE

The global-attention layers keep low-rank query and KV projections and use no positional encoding on this path.


In [ ]:
class TinyGatedMLA(nn.Module):
    def __init__(
        self,
        model_dim=8,
        heads=2,
        q_rank=4,
        kv_rank=4,
    ):
        super().__init__()
        assert model_dim % heads == 0

        self.model_dim = model_dim
        self.heads = heads
        self.head_dim = model_dim // heads

        self.q_down = nn.Linear(model_dim, q_rank, bias=False)
        self.q_up = nn.Linear(q_rank, model_dim, bias=False)
        self.kv_down = nn.Linear(model_dim, kv_rank, bias=False)
        self.kv_up = nn.Linear(kv_rank, 2 * model_dim, bias=False)

        self.output_gate = nn.Linear(model_dim, model_dim, bias=True)
        self.out_proj = nn.Linear(model_dim, model_dim, bias=False)

    def _split_heads(self, x):
        batch, length, _ = x.shape
        return x.view(
            batch, length, self.heads, self.head_dim
        ).transpose(1, 2)

    def forward(self, hidden):
        q = self._split_heads(self.q_up(self.q_down(hidden)))

        k, v = self.kv_up(self.kv_down(hidden)).chunk(2, dim=-1)
        k = self._split_heads(k)
        v = self._split_heads(v)

        attended = F.scaled_dot_product_attention(
            q.float(),
            k.float(),
            v.float(),
            is_causal=True,
        ).to(hidden.dtype)
        attended = attended.transpose(1, 2).contiguous().reshape_as(hidden)

        gate = torch.sigmoid(self.output_gate(hidden))
        return self.out_proj(gate * attended)


## 4. Block Attention Residuals

A layer-specific pseudo-query scores the banked block representations and the current partial residual after RMS normalization. At a 12-layer boundary, the raw incoming residual is banked and the new block begins from the attention output. Attention and FFN each have their own pseudo-query.


In [ ]:
class AttnResMix(nn.Module):
    def __init__(self, model_dim):
        super().__init__()
        self.score = nn.Parameter(torch.randn(model_dim) * 0.02)
        self.norm = RMSNorm(model_dim)

    def forward(self, current, checkpoints):
        if not checkpoints:
            return current

        sources = checkpoints + [current]
        stacked = torch.stack(sources, dim=2)
        scores = (self.norm(stacked) * self.score).sum(dim=-1)
        weights = scores.softmax(dim=-1)
        return (stacked * weights[..., None]).sum(dim=2)


class SiTUFFN(nn.Module):
    def __init__(self, model_dim=8, hidden_dim=16):
        super().__init__()
        self.gate = nn.Linear(model_dim, hidden_dim, bias=False)
        self.value = nn.Linear(model_dim, hidden_dim, bias=False)
        self.out = nn.Linear(hidden_dim, model_dim, bias=False)

    def forward(self, hidden):
        gate_pre = self.gate(hidden)
        value_pre = self.value(hidden)

        gate = 4.0 * torch.tanh(gate_pre / 4.0) * torch.sigmoid(gate_pre)
        value = 25.0 * torch.tanh(value_pre / 25.0)
        return self.out(gate * value)


class K3AttentionLayer(nn.Module):
    def __init__(self, kind, model_dim=8):
        super().__init__()
        self.attn_mix = AttnResMix(model_dim)
        self.ffn_mix = AttnResMix(model_dim)
        self.attn_norm = RMSNorm(model_dim)
        self.ffn_norm = RMSNorm(model_dim)

        if kind == "kda":
            self.attention = TinyKDA(model_dim=model_dim)
        elif kind == "mla":
            self.attention = TinyGatedMLA(model_dim=model_dim)
        else:
            raise ValueError(kind)

        self.ffn = SiTUFFN(model_dim=model_dim)

    def attention_output(self, prefix_sum, checkpoints):
        mixed = self.attn_mix(prefix_sum, checkpoints)
        return self.attention(self.attn_norm(mixed))

    def ffn_output(self, prefix_sum, checkpoints):
        mixed = self.ffn_mix(prefix_sum, checkpoints)
        return self.ffn(self.ffn_norm(mixed))


## 5. The full 93-layer K3 attention schedule

The schedule contains exactly 69 KDA and 24 MLA layers. Every fourth layer is MLA, with the final layer also global MLA.


In [ ]:
class K3AttentionSpine(nn.Module):
    def __init__(
        self,
        model_dim=8,
        num_layers=93,
        attn_res_block_size=12,
    ):
        super().__init__()
        self.attn_res_block_size = attn_res_block_size

        self.kinds = [
            "mla"
            if ((layer_index + 1) % 4 == 0 or layer_index == num_layers - 1)
            else "kda"
            for layer_index in range(num_layers)
        ]
        self.layers = nn.ModuleList(
            [K3AttentionLayer(kind, model_dim) for kind in self.kinds]
        )
        self.output_mix = AttnResMix(model_dim)

    def forward(self, hidden):
        checkpoints = []
        prefix_sum = hidden

        for layer_index, layer in enumerate(self.layers):
            attention_output = layer.attention_output(
                prefix_sum,
                checkpoints,
            )

            starts_new_block = (
                layer_index % self.attn_res_block_size == 0
            )
            if starts_new_block:
                checkpoints.append(prefix_sum)
                prefix_sum = attention_output
            else:
                prefix_sum = prefix_sum + attention_output

            prefix_sum = prefix_sum + layer.ffn_output(
                prefix_sum,
                checkpoints,
            )

        return self.output_mix(prefix_sum, checkpoints)


model = K3AttentionSpine().to(device)
print("layers:", len(model.layers))
print("KDA layers:", model.kinds.count("kda"))
print("MLA layers:", model.kinds.count("mla"))


## 6. CPU five-step backward sanity check

This is deliberately not a quality-training experiment. It checks that all 93 layers participate in one differentiable graph and that a tiny optimization problem can move the loss.


In [ ]:
hidden = torch.randn(1, 3, 8, device=device)
target = torch.zeros_like(hidden)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

loss_history = []
for step in range(5):
    optimizer.zero_grad()
    prediction = model(hidden)
    loss = prediction.square().mean()
    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())
    print(f"step {step + 1}: loss={loss.item():.6f}")

print("loss history:", loss_history)


## References and provenance

- Moonshot AI, Kimi K3 technical report: KDA, Gated MLA, 93-layer 69:24 schedule, and Block AttnRes.
- K3 released/config-derived implementation details: `attn_res_block_size=12`, KDA `f_a/f_b` low-rank decay path, `gate_lower_bound=-5`, and full-rank output gate.

Only tensor dimensions and the training budget are reduced here.
